# ⚡ DM AI OS v1.5.1 — Google Colab Tesla T4 Compute Worker
### Instrucciones

**Paso 1:** Asegúrate de tener seleccionado **GPU T4** como acelerador.
- Menú → Entorno de ejecución → Cambiar tipo → GPU T4

**Paso 2:** Ejecuta **Celda 1** para montar Google Drive con la cuenta que tiene los modelos.
- Cuando Google pregunte, selecciona la cuenta correcta (la de 5 TB) y acepta los permisos.
- Ninguna contraseña se almacena en este notebook ni en el repositorio.

**Paso 3:** Ejecuta **Celda 2** para verificar el Model Storage.
- Esta celda NO descarga ningún modelo.
- Solo verifica que la estructura `DM-AI-OS-MODELS/` exista y sea accesible.

**Paso 4:** Ejecuta **Celda 3** para arrancar el Compute Worker completo.
- El bootstrap descubrirá automáticamente los modelos disponibles en Drive.
- Solo registrará capacidades reales (no estáticas).
- SD15 continuará funcionando E2E durante todo el proceso.

In [ ]:
#@title 1️⃣ Montar Google Drive (OAuth — sin contraseñas) { display-mode: "form" }
#
# INSTRUCCIONES:
# 1. Haz clic en ▶ (Play)
# 2. Cuando aparezca el popup de autorización, selecciona la cuenta
#    que contiene la carpeta DM-AI-OS-MODELS (cuenta con Google One 5 TB)
# 3. Acepta los permisos de Google Drive
# 4. Espera a que aparezca: 'Mounted at /content/drive'
#
# SEGURIDAD: Ningún token ni contraseña se almacena en el código.
# La sesión OAuth de Colab expira al reiniciar el runtime.
#
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive montado en /content/drive')

In [ ]:
#@title 2️⃣ Verificar Model Storage (DM-AI-OS-MODELS) { display-mode: "form" }
#
# Esta celda SOLO verifica — NO descarga modelos.
# Muestra el estado de la estructura de carpetas y acceso en Drive.
# El espacio disponible se muestra como INFORMATIVO (no es verificación de cuota).
#
import os, shutil
from pathlib import Path

# Override via env var for flexibility (no hardcoded paths to any account)
MODELS_ROOT = Path(os.getenv('DM_DRIVE_MODELS_PATH', '/content/drive/MyDrive/DM-AI-OS-MODELS'))

print('=' * 60)
print('🔍 DM AI OS — Model Storage Diagnostic')
print(f'   Root: {MODELS_ROOT}')
print('=' * 60)

# DRIVE MOUNT
drive_ok = Path('/content/drive/MyDrive').exists()
print(f'DRIVE MOUNT:                {"✅ PASS" if drive_ok else "❌ FAIL — ejecuta Celda 1"}')

# MODEL STORAGE ROOT
root_ok = MODELS_ROOT.exists()
print(f'MODEL STORAGE ROOT:         {"✅ PASS" if root_ok else "❌ FAIL — crear DM-AI-OS-MODELS en Drive"}')

if not root_ok:
    print('\n⚠️ Crear manualmente en Drive:')
    print('   Mi Unidad → DM-AI-OS-MODELS → subcarpetas: checkpoints, diffusion_models, clip, vae, loras')
    raise SystemExit('Crear estructura de carpetas en Drive primero.')

# READ ACCESS
read_ok = False
try:
    list(MODELS_ROOT.iterdir())
    read_ok = True
except Exception as e:
    print(f'   Read error: {e}')
print(f'READ ACCESS:                {"✅ PASS" if read_ok else "❌ FAIL"}')

# WRITE ACCESS (temp file only)
write_ok = False
test_file = MODELS_ROOT / '.dm_write_test'
try:
    test_file.write_text('DM-AI-OS write test')
    test_file.unlink()
    write_ok = True
except Exception as e:
    print(f'   Write error: {e}')
print(f'WRITE ACCESS:               {"✅ PASS" if write_ok else "❌ FAIL"}')

# REPORTED AVAILABLE STORAGE (informational only — not 5TB verification)
try:
    total, used, free = shutil.disk_usage('/content/drive/MyDrive')
    free_gb = round(free / (1024**3), 1)
    print(f'REPORTED AVAILABLE STORAGE: ℹ️  {free_gb} GB (OS-reported, informational only)')
except Exception:
    print('REPORTED AVAILABLE STORAGE: ℹ️  No disponible')

# DIRECTORY STRUCTURE
expected_dirs = ['checkpoints', 'diffusion_models', 'text_encoders', 'clip', 'vae', 'loras', 'controlnet', 'upscale_models', 'manifests']
missing = [d for d in expected_dirs if not (MODELS_ROOT / d).exists()]
found = len(expected_dirs) - len(missing)
print(f'DIRECTORIES:                {"✅ PASS" if not missing else "⚠️ PARTIAL"} ({found}/{len(expected_dirs)})')
if missing:
    print(f'   Faltantes: {missing}')
    print('   Créalas en Drive antes de subir modelos.')

# SCAN EXISTING MODEL FILES
print('\n📦 Archivos encontrados en Model Storage:')
total_files = 0
for cat in ['checkpoints', 'diffusion_models', 'clip', 'vae', 'loras']:
    cat_path = MODELS_ROOT / cat
    if cat_path.exists():
        files = [f for f in cat_path.iterdir() if f.suffix in ('.safetensors', '.gguf', '.bin', '.pt')]
        if files:
            print(f'   {cat}/')
            for f in files:
                size_gb = round(f.stat().st_size / (1024**3), 2)
                print(f'      • {f.name} ({size_gb} GB)')
            total_files += len(files)

if total_files == 0:
    print('   (No se encontraron archivos de modelos todavía)')
    print('   Sube modelos manualmente o mediante atajos compartidos de Drive.')

print('\n' + '=' * 60)
print(f'OVERALL: {"✅ STORAGE READY" if (drive_ok and root_ok and read_ok and write_ok) else "⚠️ ACCIÓN REQUERIDA"}')
print('=' * 60)

In [ ]:
#@title 3️⃣ Iniciar DM AI OS Compute Worker { display-mode: "form" }
#
# El bootstrap descubrirá automáticamente los modelos disponibles en Drive.
# Solo registrará como capabilidades los modelos físicamente verificados.
# SD15 continuará funcionando E2E (no se modifica su workflow).
# FLUX quedará en estado CONFIGURED — requiere prueba física para READY.
#
import os

# Configuration (no credentials — identity resolved by OAuth above)
os.environ['DM_AI_OS_URL'] = 'https://ai.dmorales.com.ar'
os.environ['DM_WORKER_ID'] = 'colab-comfy-primary'
# DM_DRIVE_MODELS_PATH is auto-resolved if not set:
# Default: /content/drive/MyDrive/DM-AI-OS-MODELS
# Override: os.environ['DM_DRIVE_MODELS_PATH'] = '/content/drive/MyDrive/YourCustomPath'

print('📥 Obteniendo bootstrap desde GitHub main...')
!wget -q https://raw.githubusercontent.com/daniel2029m-droid/dm-ai-os/main/deployment/colab_bootstrap.py -O /content/colab_bootstrap.py

print('🚀 Ejecutando DM AI OS Compute Worker Bootstrap...')
!python /content/colab_bootstrap.py